In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

25/08/04 13:54:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Read Data


In [2]:
medals = spark.read.option("header", "true").csv("/home/iceberg/data/medals.csv")
maps = spark.read.option("header", "true").csv("/home/iceberg/data/maps.csv")
matches = spark.read.option("header", "true").csv("/home/iceberg/data/matches.csv")
medals_matches_players = spark.read.option("header", "true").csv("/home/iceberg/data/medals_matches_players.csv")
match_details = spark.read.option("header", "true").csv("/home/iceberg/data/match_details.csv")

print("Medals DataFrame Schema:")
medals.printSchema()
print("Total Rows:", medals.count())
medals.show(5)
print("=" * 300)


print("Maps DataFrame Schema:")
maps.printSchema()
print("Total Rows:", maps.count())
maps.show(5)
print("=" * 300)

print("Matches DataFrame Schema:")
matches.printSchema()
print("Total Rows:", matches.count())
matches.show(5)
print("=" * 300)

print("Medals Matches Players DataFrame Schema:")
medals_matches_players.printSchema()
print("Total Rows:", medals_matches_players.count())
medals_matches_players.show(5)
print("=" * 300)

print("Match Details DataFrame Schema:")
match_details.printSchema()
print("Total Rows:", match_details.count())
match_details.show(5)
print("=" * 300)

Medals DataFrame Schema:
root
 |-- medal_id: string (nullable = true)
 |-- sprite_uri: string (nullable = true)
 |-- sprite_left: string (nullable = true)
 |-- sprite_top: string (nullable = true)
 |-- sprite_sheet_width: string (nullable = true)
 |-- sprite_sheet_height: string (nullable = true)
 |-- sprite_width: string (nullable = true)
 |-- sprite_height: string (nullable = true)
 |-- classification: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- difficulty: string (nullable = true)

Total Rows: 183
+----------+--------------------+-----------+----------+------------------+-------------------+------------+-------------+--------------+--------------------+--------------+----------+
|  medal_id|          sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|         description|          name|difficulty|
+----------+--------------------+-----------+----------+---

25/08/04 13:54:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Create bucketed tables



In [3]:
# Check existing databases and catalog configuration
print("Existing databases:")
spark.sql("SHOW DATABASES").show()

print("\nBootcamp tables (if any):")
try:
    spark.sql("SHOW TABLES IN bootcamp").show()
except Exception as e:
    print(f"Error showing tables: {e}")

print("\nCatalog configuration:")
print("Current catalog:", spark.catalog.currentCatalog())
print("Current database:", spark.catalog.currentDatabase())

print("\nSpark SQL configurations related to Iceberg:")
for key in ["spark.sql.catalog.spark_catalog", "spark.sql.catalog.spark_catalog.type", "spark.sql.extensions"]:
    try:
        value = spark.conf.get(key)
        print(f"{key}: {value}")
    except:
        print(f"{key}: NOT SET")

print("\nCreating/recreating bootcamp database and tables...")

Existing databases:
+---------+
|namespace|
+---------+
| bootcamp|
+---------+


Bootcamp tables (if any):
+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| bootcamp|                maps|      false|
| bootcamp|       match_details|      false|
| bootcamp|match_details_buc...|      false|
| bootcamp|             matches|      false|
| bootcamp|    matches_bucketed|      false|
| bootcamp|        matches_full|      false|
| bootcamp|              medals|      false|
| bootcamp|medals_matches_pl...|      false|
| bootcamp|medals_matches_pl...|      false|
+---------+--------------------+-----------+


Catalog configuration:
Current catalog: demo
Current database: 

Spark SQL configurations related to Iceberg:
spark.sql.catalog.spark_catalog: None
spark.sql.catalog.spark_catalog.type: NOT SET
spark.sql.extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions

Creating/recreating bo

In [4]:
def create_tables(spark):
    
    # First create the bootcamp database/namespace if it doesn't exist
    spark.sql("CREATE DATABASE IF NOT EXISTS bootcamp")

    spark.sql("""
    CREATE OR REPLACE TABLE bootcamp.medals (
        medal_id STRING,
        sprite_uri STRING,
        sprite_left STRING,
        sprite_top STRING,
        sprite_sheet_width STRING,
        sprite_sheet_height STRING,
        sprite_width STRING,
        sprite_height STRING,
        classification STRING,
        description STRING,
        name STRING,
        difficulty STRING
    )
    USING iceberg
    """)

    spark.sql("""
    CREATE OR REPLACE TABLE bootcamp.maps (
        mapid STRING,
        map_name STRING,
        map_description STRING
    )
    USING iceberg
    """)

    # Bucketed version of matches table
    spark.sql("""
    CREATE OR REPLACE TABLE bootcamp.matches (
        match_id STRING,
        mapid STRING,
        is_team_game BOOLEAN,
        playlist_id STRING,
        game_variant_id STRING,
        is_match_over BOOLEAN,
        completion_date TIMESTAMP,
        match_duration STRING,
        game_mode STRING,
        map_variant_id STRING
    )
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    """)
    
    # Bucketed version of match_details table - exclude match_id to avoid duplicates
    spark.sql("""
    CREATE OR REPLACE TABLE bootcamp.match_details (
        match_id STRING,
        player_gamertag STRING,
        previous_spartan_rank STRING,
        spartan_rank STRING,
        previous_total_xp STRING,
        total_xp STRING,
        previous_csr_tier STRING,
        previous_csr_designation STRING,
        previous_csr STRING,
        previous_csr_percent_to_next_tier STRING,
        previous_csr_rank STRING,
        current_csr_tier STRING,
        current_csr_designation STRING,
        current_csr STRING,
        current_csr_percent_to_next_tier STRING,
        current_csr_rank STRING,
        player_rank_on_team STRING,
        player_finished STRING,
        player_average_life STRING,
        player_total_kills STRING,
        player_total_headshots STRING,
        player_total_weapon_damage STRING,
        player_total_shots_landed STRING,
        player_total_melee_kills STRING,
        player_total_melee_damage STRING,
        player_total_assassinations STRING,
        player_total_ground_pound_kills STRING,
        player_total_shoulder_bash_kills STRING,
        player_total_grenade_damage STRING,
        player_total_power_weapon_damage STRING,
        player_total_power_weapon_grabs STRING,
        player_total_deaths STRING,
        player_total_assists STRING,
        player_total_grenade_kills STRING,
        did_win BOOLEAN,
        team_id STRING
    )
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    """)
    
    # Bucketed version of medals_matches_players table
    spark.sql("""
    CREATE OR REPLACE TABLE bootcamp.medals_matches_players (
        match_id STRING,
        player_gamertag STRING,
        medal_id STRING,
        medal_count STRING
    )
    USING iceberg
    PARTITIONED BY (bucket(16, match_id))
    """)

In [5]:
create_tables(spark)

In [6]:
%%sql
-- Verify the tables were created
SHOW TABLES IN bootcamp;

namespace,tableName,isTemporary
bootcamp,maps,False
bootcamp,match_details,False
bootcamp,match_details_bucketed,False
bootcamp,matches,False
bootcamp,matches_bucketed,False
bootcamp,matches_full,False
bootcamp,medals,False
bootcamp,medals_matches_players,False
bootcamp,medals_matches_players_bucketed,False


## Load the tables

Load each iceberg table with the data from CSVs

In [7]:
def load_tables(spark):
    medals_ = spark.read.option("header", "true").csv("/home/iceberg/data/medals.csv")
    maps_ = spark.read.option("header", "true").csv("/home/iceberg/data/maps.csv")
    matches_ = spark.read.option("header", "true").csv("/home/iceberg/data/matches.csv")
    medals_matches_players_ = spark.read.option("header", "true").csv("/home/iceberg/data/medals_matches_players.csv")
    match_details_ = spark.read.option("header", "true").csv("/home/iceberg/data/match_details.csv")

    # Rename columns to avoid duplicates
    maps_renamed = maps_.withColumnRenamed("name", "map_name").withColumnRenamed("description", "map_description")
    medals_matches_players_renamed = medals_matches_players_.withColumnRenamed("count", "medal_count")

    medals_.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.medals")
    maps_renamed.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.maps")
    matches_.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.matches")
    match_details_.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.match_details")
    medals_matches_players_renamed.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.medals_matches_players")

    return True

In [8]:
load_tables(spark)

True

In [9]:
%%sql

select * from bootcamp.medals limit 10;

medal_id,sprite_uri,sprite_left,sprite_top,sprite_sheet_width,sprite_sheet_height,sprite_width,sprite_height,classification,description,name,difficulty
2315448068,None,None,None,None,None,None,None,None,None,None,None
3565441934,None,None,None,None,None,None,None,None,None,None,None
4162659350,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,750,750,74,74,1125,899,Breakout,Kill the last enemy within the last 10 seconds of a round.,Buzzer Beater,45
1573153198,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,0,300,74,74,1125,899,Breakout,Survive a one-on-one encounter.,Vanquisher,30
298813630,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,0,825,74,74,1125,899,Style,Kill an enemy with Spartan Charge.,Spartan Charge,135
3824002610,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,750,600,74,74,1125,899,Vehicles,Assist a player in destroying an enemy Ghost.,Ghost Assist,140
3324603383,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,225,750,74,74,1125,899,Warzone,Kill a Grunt.,Grunt Kill,60
979431049,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,975,675,74,74,1125,899,Breakout,Survive a two-on-one encounter.,Bifecta,25
3098362934,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,150,750,74,74,1125,899,WeaponProficiency,Kill a player in seven shots with the Carbine without missing.,Perfect Kill,25
2435743433,https://content.halocdn.com/media/Default/games/halo-5-guardians/sprites/medalspritesheet-be288ea5c0994a4e9d36f43aee7bc631.png,750,225,74,74,1125,899,Warzone,Kill an enemy in a controlled Base.,Base Defense,30


## Join Tables

Join matches -> (broadcast) maps -> (bucket) matches_players_medals -> (broadcast) medals

In [10]:
from pyspark.sql.functions import broadcast


def do_joins(spark):

    medals = spark.table("bootcamp.medals")
    maps = spark.table("bootcamp.maps")
    matches = spark.table("bootcamp.matches")
    medals_matches_players = spark.table("bootcamp.medals_matches_players")
    match_details = spark.table("bootcamp.match_details")

    # Start with match_details as it has both match_id and player_gamertag
    result = match_details
    
    # Join with matches on match_id to get match-level information
    result = result.join(matches, "match_id", "left")
    
    # Join with maps using mapid (broadcast for small table)
    result = result.join(broadcast(maps), "mapid", "left")

    # Join with medals_matches_players on both match_id and player_gamertag
    result = result.join(medals_matches_players, ["match_id", "player_gamertag"], "left")

    # Finally join with medals using medal_id (broadcast for small table)
    result = result.join(broadcast(medals), "medal_id", "left")

    return result

In [11]:
result_df = do_joins(spark) 
result_df.show(5)

+----------+--------------------+---------------+--------------------+---------------------+------------+-----------------+--------+-----------------+------------------------+------------+---------------------------------+-----------------+----------------+-----------------------+-----------+--------------------------------+----------------+-------------------+---------------+-------------------+------------------+----------------------+--------------------------+-------------------------+------------------------+-------------------------+---------------------------+-------------------------------+--------------------------------+---------------------------+--------------------------------+-------------------------------+-------------------+--------------------+--------------------------+-------+-------+------------+--------------------+--------------------+-------------+--------------------+--------------+---------+--------------------+--------+--------------------+-----------+-------

In [12]:
## Write the result to a new Iceberg table

result_df.write.format("iceberg").mode("overwrite").saveAsTable("bootcamp.matches_full")

## Analyse the data

1. Which player averages the most kills per game?
1. Which playlist gets played the most?
1. Which map gets played the most?
1. Which map do players get the most Killing Spree medals on?

In [16]:
# Final Analysis Summary - Clean Results
from pyspark.sql.functions import col, sum as spark_sum

print("=" * 60)
print("HALO MATCH ANALYSIS RESULTS")
print("=" * 60)

# 1. Player with highest average kills per game
top_killer = result_df.groupBy("player_gamertag") \
    .agg({"player_total_kills": "avg"}) \
    .orderBy("avg(player_total_kills)", ascending=False) \
    .first()

if top_killer:
    print(f"1. Player with highest average kills per game: {top_killer['player_gamertag']}")
else:
    print("1. No player data found")

# 2. Most played playlist
top_playlist = result_df.groupBy("playlist_id") \
    .count() \
    .orderBy("count", ascending=False) \
    .first()

if top_playlist:
    print(f"2. Most played playlist: {top_playlist['playlist_id']}")
else:
    print("2. No playlist data found")

# 3. Most played map
top_map = result_df.groupBy("map_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .first()

if top_map:
    print(f"3. Most played map: {top_map['map_name']}")
else:
    print("3. No map data found")

# 4. Map with most Killing Spree medals (using the corrected logic)
top_killing_spree_map = result_df.filter(col("classification").contains("KillingSpree")).groupBy("map_name") \
 .agg(spark_sum("medal_count").alias("total_medals")) \
 .orderBy("total_medals", ascending=False) \
 .first()

if top_killing_spree_map:
    print(f"4. Map with most Killing Spree medals: {top_killing_spree_map['map_name']}")
else:
    print("4. No Killing Spree medal data found")

# Additional insights
# 5. Player with highest total weapon damage
top_damage_player = result_df.groupBy("player_gamertag") \
    .agg({"player_total_weapon_damage": "sum"}) \
    .orderBy("sum(player_total_weapon_damage)", ascending=False) \
    .first()

if top_damage_player:
    print(f"5. Player with highest total weapon damage: {top_damage_player['player_gamertag']}")

# 6. Most frequently awarded medal
top_medal = result_df.groupBy("medal_id", "name") \
    .agg(spark_sum("medal_count").alias("total_count")) \
    .orderBy("total_count", ascending=False) \
    .first()

if top_medal:
    print(f"6. Most frequently awarded medal: {top_medal['name']}")

print("=" * 60)
print("Analysis complete!")

HALO MATCH ANALYSIS RESULTS
1. Player with highest average kills per game: gimpinator14
1. Player with highest average kills per game: gimpinator14
2. Most played playlist: f72e0ef0-7c4a-4307-af78-8e38dac3fdba
2. Most played playlist: f72e0ef0-7c4a-4307-af78-8e38dac3fdba
3. Most played map: Breakout Arena
3. Most played map: Breakout Arena
4. Map with most Killing Spree medals: Breakout Arena
4. Map with most Killing Spree medals: Breakout Arena
5. Player with highest total weapon damage: EcZachly
5. Player with highest total weapon damage: EcZachly
6. Most frequently awarded medal: Headshot
Analysis complete!
6. Most frequently awarded medal: Headshot
Analysis complete!
